# UNIPLU-BR Methodology Tutorial

This notebook documents the official academic workflow for processing UNIPLU-BR rainfall observations and generating hydroclimatic products for MATLAB dashboard analysis.

Project goals:
- convert daily rainfall records into quality-controlled monthly totals,
- preserve geospatial metadata for station-based interpretation,
- export reproducible outputs to CSV and MAT formats.

## Hydrological Theory: Missing Data and Monthly Aggregation

Rainfall datasets frequently contain missing daily observations. If monthly totals are computed without quality control, under-sampled months can appear artificially dry, producing false dry-month anomalies in historical series.

To mitigate this bias, we apply a strict hydrological threshold:
- a month is considered valid only when daily completeness is at least 90%,
- equivalently, missing daily records must not exceed 10% of expected calendar days.

This rule improves the reliability of anomaly diagnostics and non-parametric trend analyses.

In [2]:
import pandas as pd
from scipy.io import savemat
from pathlib import Path
import zipfile

root = Path('.')
daily_csv = root / 'df_cemaden_daily.csv'
info_csv = root / 'df_total_info.csv'

def read_uniplu_br(zip_path: Path, table: str) -> pd.DataFrame:
    """Read one parquet table from a UNIPLU-BR ZIP package."""
    parquet_name = f"{table}.parquet"
    with zipfile.ZipFile(zip_path, 'r') as zf:
        with zf.open(parquet_name) as f:
            return pd.read_parquet(f)

if daily_csv.exists() and info_csv.exists():
    # Preferred path when explicit raw CSV exports already exist.
    df_daily = pd.read_csv(daily_csv, parse_dates=['datetime'])
    df_info = pd.read_csv(info_csv)
else:
    # Fallback for this repository layout: load raw data directly from ZIP/parquet files.
    data_dir = root / 'UNIPLU_BR-dados'
    zip_paths = [
        data_dir / 'RS_2023.zip',
        data_dir / 'RS_2024.zip',
        data_dir / 'SC_2023.zip',
        data_dir / 'SC_2024.zip',
    ]

    existing = [p for p in zip_paths if p.exists()]
    if not existing:
        raise FileNotFoundError(
            'No raw CSV files were found and no fallback ZIP files exist. '
            'Expected df_cemaden_daily.csv/df_total_info.csv or RS_2023.zip, RS_2024.zip, SC_2023.zip, SC_2024.zip.'
        )

    data_frames = [read_uniplu_br(p, 'table_data') for p in existing]
    info_frames = [read_uniplu_br(p, 'table_info') for p in existing]

    df_daily = pd.concat(data_frames, ignore_index=True)
    df_info = pd.concat(info_frames, ignore_index=True).drop_duplicates(subset=['gauge_code'])

# Standardize expected columns/types for downstream notebook cells.
df_daily['datetime'] = pd.to_datetime(df_daily['datetime'], errors='coerce')
df_daily['rain_mm'] = pd.to_numeric(df_daily['rain_mm'], errors='coerce')

print(f"Loaded daily rows: {len(df_daily):,}")
print(f"Loaded station metadata rows: {len(df_info):,}")
df_daily.head()

Loaded daily rows: 19,336,202
Loaded station metadata rows: 1,412


,gauge_code,rain_mm,datetime
0,A801,0.0,2023-01-01 00:00:00
1,A801,0.0,2023-01-01 01:00:00
2,A801,0.0,2023-01-01 02:00:00
3,A801,0.0,2023-01-01 03:00:00
4,A801,0.0,2023-01-01 04:00:00


In [ ]:
import calendar

def aggregate_daily_to_monthly_strict(df_daily: pd.DataFrame, min_completeness: float = 0.90) -> pd.DataFrame:
    """Aggregate daily rainfall to monthly totals with strict 90% completeness.

    Hydrological logic:
    1) Group records by gauge_code/year/month.
    2) Compute expected daily count from calendar month length.
    3) Accept monthly total only when completeness >= 90%.
    """
    df = df_daily.copy()
    df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
    df['rain_mm'] = pd.to_numeric(df['rain_mm'], errors='coerce')
    df = df.dropna(subset=['datetime'])

    df['year'] = df['datetime'].dt.year
    df['month'] = df['datetime'].dt.month

    monthly = (
        df.groupby(['gauge_code', 'year', 'month'], as_index=False)
          .agg(
              rain_mm_monthly_sum=('rain_mm', 'sum'),
              n_days_with_data=('rain_mm', 'count')
          )
    )

    monthly['n_days_expected'] = monthly.apply(
        lambda r: calendar.monthrange(int(r['year']), int(r['month']))[1],
        axis=1
    )
    monthly['completeness'] = monthly['n_days_with_data'] / monthly['n_days_expected']
    monthly['is_valid_month'] = monthly['completeness'] >= min_completeness

    monthly['rain_mm'] = monthly['rain_mm_monthly_sum'].where(monthly['is_valid_month'], pd.NA)
    monthly['datetime'] = pd.to_datetime(dict(year=monthly['year'], month=monthly['month'], day=1))

    return monthly[['gauge_code', 'datetime', 'year', 'month', 'rain_mm', 'completeness', 'is_valid_month']]

df_monthly = aggregate_daily_to_monthly_strict(df_daily)
df_monthly.head()

In [ ]:
# Keep only valid monthly records for hydroclimatic analysis.
df_monthly_filtered = df_monthly[df_monthly['is_valid_month']].copy()

# Merge with station metadata for geographic visualization.
metadata_cols = ['gauge_code', 'lat', 'long', 'city', 'state', 'network']
df_info_unique = df_info[metadata_cols].drop_duplicates(subset=['gauge_code'])
df_monthly_with_geo = df_monthly_filtered.merge(df_info_unique, on='gauge_code', how='left', validate='m:1')

# Export MATLAB-ready structure.
output_mat = Path('dados_hidro_br_mensal_real.mat')
mat_struct = {c: df_monthly_filtered[c].astype('string').fillna('').to_numpy(dtype=object) if df_monthly_filtered[c].dtype == 'object' else pd.to_numeric(df_monthly_filtered[c], errors='coerce').to_numpy() for c in ['gauge_code', 'year', 'month', 'rain_mm']}
savemat(output_mat, {'dados_hidro_br_mensal': mat_struct})

# Export CSV products.
Path('outputs').mkdir(exist_ok=True)
df_monthly_filtered.to_csv('outputs/df_monthly_filtered_real.csv', index=False)
df_monthly_with_geo.to_csv('outputs/df_monthly_filtered_with_geo_real.csv', index=False)

print('Saved outputs/df_monthly_filtered_real.csv')
print('Saved outputs/df_monthly_filtered_with_geo_real.csv')
print('Saved dados_hidro_br_mensal_real.mat')

## Data Quality Verification

After filtering, data availability should be visualized before trend interpretation. Availability diagnostics reveal temporal gaps, station discontinuities, and potential sampling bias.

A heatmap (gauge_code x datetime) supports rapid inspection of valid versus missing monthly records and documents the effective analysis domain.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv('outputs/df_monthly_filtered_with_geo_real.csv')

# Build datetime axis from year/month to preserve monthly sequence.
df['datetime'] = pd.to_datetime(dict(year=df['year'].astype(int), month=df['month'].astype(int), day=1))

# Availability matrix: 1 for valid rain_mm, 0 for missing rain_mm.
df['available'] = df['rain_mm'].notna().astype(int)
availability = df.pivot_table(index='gauge_code', columns='datetime', values='available', aggfunc='max')

# Gantt-like availability heatmap (green = valid, red = missing).
plt.figure(figsize=(18, 10))
sns.heatmap(
    availability.fillna(0),
    cmap=sns.color_palette(['#c62828', '#2e7d32']),
    cbar=True,
    vmin=0,
    vmax=1
)
plt.title('UNIPLU-BR Monthly Data Availability (Gauge x Time)')
plt.xlabel('Datetime')
plt.ylabel('Gauge Code')
plt.tight_layout()
plt.show()